# A3 — Turnover Sample Attrition Characterisation (13,467 → 446 functions)

**Reviewer concern addressed:** Section A3 of the revision checklist — *"A 96.7% drop in sample size is the biggest validity threat in the paper and receives only one brief sentence. Add a table showing attrition reasons broken down by agent vs. developer cohort. If attrition is non-random and asymmetric, the turnover conclusions may not generalise."*

This notebook:
1. **Attrition funnel** — tracks how many functions are lost at each filtering stage, broken down by `author_type`.
2. **Survivorship bias check** — compares the complexity/documentation profile of retained vs. excluded functions.
3. **Commit velocity control** — computes median time-between-commits for agent vs. developer repositories.
4. Saves all outputs to `revision_outputs/attrition_*.csv`.

In [ ]:
import pandas as pd
import numpy as np
import ast
import re
import os
import warnings
warnings.filterwarnings('ignore')

import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

sns.set_theme(style='whitegrid')
PALETTE = {'Agent': '#A7C7E7', 'Developer': '#BDE5B8'}

DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), 'dataset', 'data', 'updated_dataset_metrics.csv')
OUT_DIR   = os.path.join(os.getcwd(), 'revision_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

# ── Load raw (no filtering) ───────────────────────────────────────────────────
df_raw = pd.read_csv(DATA_PATH)

numeric_cols = [
    'doc_lines', 'doc_entropy', 'doc_code_overlap', 'doc_redundancy',
    'cyclomatic_complexity', 'sloc', 'semgrep_findings_count', 'num_parameters'
]
for col in numeric_cols:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

print(f"Raw dataset shape: {df_raw.shape}")
print(f"Group counts:\n{df_raw['group'].value_counts().to_string()}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TURNOVER EXTRACTION HELPER
# ─────────────────────────────────────────────────────────────────────────────

def extract_turnover(val):
    """Extract numeric turnover value from string-encoded tuple or raw float."""
    if pd.isna(val) or val == -1 or val == '-1':
        return np.nan
    try:
        if isinstance(val, str) and '(' in val:
            parsed = ast.literal_eval(val)
            val = parsed[1]
        num = float(val)
        return num if num >= 0 else np.nan
    except Exception:
        return np.nan

TURNOVER_COLS = ['turnover_c5', 'turnover_c10', 'turnover_c20', 'turnover_m1', 'turnover_m3']

df_raw['_c5_raw'] = df_raw['turnover_c5'].apply(extract_turnover)

for col in TURNOVER_COLS:
    df_raw[f'_{col}_num'] = df_raw[col].apply(extract_turnover)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. ATTRITION FUNNEL
#    Stage 0: All functions in dataset
#    Stage 1: After dropping functions without any documentation
#    Stage 2: After requiring at least one non-NaN turnover_c5 value
#    Stage 3: After requiring ALL turnover columns to be non-NaN
# ─────────────────────────────────────────────────────────────────────────────
print('=== ATTRITION FUNNEL ===')

funnel_rows = []

def funnel_step(label, mask_series):
    """Record a funnel step for both cohorts."""
    sub = df_raw[mask_series]
    total = len(sub)
    n_ag  = (sub['group'] == 'agent').sum()
    n_hu  = (sub['group'] == 'human').sum()
    funnel_rows.append({
        'stage': label,
        'n_total': total,
        'n_agent': n_ag,
        'n_developer': n_hu,
    })
    print(f"  {label:<55} total={total:5d}  agent={n_ag:5d}  dev={n_hu:5d}")
    return sub

# Stage 0: raw
mask_s0 = pd.Series([True] * len(df_raw), index=df_raw.index)
funnel_step('S0: All functions', mask_s0)

# Stage 1: documentation quality columns present
mask_s1 = df_raw['doc_entropy'].notna() & df_raw['doc_code_overlap'].notna() & df_raw['doc_redundancy'].notna()
funnel_step('S1: Has documentation quality metrics', mask_s1)

# Stage 1b: doc_lines > 0
mask_s1b = mask_s1 & (df_raw['doc_lines'] > 0)
funnel_step('S1b: doc_lines > 0 (has actual doc)', mask_s1b)

# Stage 2: at least turnover_c5 is available
mask_s2 = mask_s1b & df_raw['_c5_raw'].notna()
funnel_step('S2: turnover_c5 is non-NaN', mask_s2)

# Stage 3: all turnover columns available
all_turn_valid = pd.Series([True] * len(df_raw), index=df_raw.index)
for col in TURNOVER_COLS:
    all_turn_valid = all_turn_valid & df_raw[f'_{col}_num'].notna()
mask_s3 = mask_s1b & all_turn_valid
funnel_step('S3: ALL turnover columns non-NaN (final RQ4 dataset)', mask_s3)

funnel_df = pd.DataFrame(funnel_rows)

# Attrition rate per cohort (relative to Stage 0)
s0_ag = funnel_df.loc[0, 'n_agent']
s0_hu = funnel_df.loc[0, 'n_developer']
funnel_df['pct_agent_retained']   = funnel_df['n_agent']     / s0_ag * 100
funnel_df['pct_developer_retained'] = funnel_df['n_developer'] / s0_hu * 100

display(funnel_df)

# ── FLAG asymmetric attrition ─────────────────────────────────────────────────
final = funnel_df.iloc[-1]
attrition_diff = abs(final['pct_agent_retained'] - final['pct_developer_retained'])
print(f"\nFinal retention: agent={final['pct_agent_retained']:.1f}%  developer={final['pct_developer_retained']:.1f}%")
if attrition_diff > 10:
    print(f"*** ASYMMETRIC ATTRITION DETECTED: {attrition_diff:.1f} percentage-point difference ***")
    print("    This threatens the validity of between-group turnover comparisons.")
    print("    Report and discuss in the Threats to Validity section.")
else:
    print(f"Attrition is roughly symmetric ({attrition_diff:.1f} pp difference). Turnover comparisons are comparably affected.")

funnel_df.to_csv(os.path.join(OUT_DIR, 'attrition_funnel.csv'), index=False)
print('Saved attrition_funnel.csv')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ATTRITION FUNNEL: per-agent breakdown
# ─────────────────────────────────────────────────────────────────────────────
print('\n=== Per-Agent Attrition ===')

per_agent_rows = []
for lbl in df_raw['label'].dropna().unique():
    sub = df_raw[df_raw['label'] == lbl]
    s0 = len(sub)
    s1 = sub['doc_entropy'].notna().sum()
    s1b = ((sub['doc_entropy'].notna()) & (sub['doc_lines'] > 0)).sum()
    s3_mask = pd.Series([True] * len(sub), index=sub.index)
    for col in TURNOVER_COLS:
        s3_mask = s3_mask & sub[f'_{col}_num'].notna()
    s3 = s3_mask.sum()
    per_agent_rows.append({
        'agent': lbl,
        'S0_total': s0,
        'S1b_has_doc': s1b,
        'S3_all_turnover': s3,
        'pct_retained_S3': 100 * s3 / s0 if s0 > 0 else 0,
        'pct_retained_S1b': 100 * s1b / s0 if s0 > 0 else 0,
    })

# Developer
sub = df_raw[df_raw['group'] == 'human']
s0, s1b = len(sub), ((sub['doc_entropy'].notna()) & (sub['doc_lines'] > 0)).sum()
s3_mask = pd.Series([True] * len(sub), index=sub.index)
for col in TURNOVER_COLS:
    s3_mask = s3_mask & sub[f'_{col}_num'].notna()
per_agent_rows.append({
    'agent': 'Developer (human)',
    'S0_total': s0, 'S1b_has_doc': s1b, 'S3_all_turnover': s3_mask.sum(),
    'pct_retained_S3': 100 * s3_mask.sum() / s0 if s0 > 0 else 0,
    'pct_retained_S1b': 100 * s1b / s0 if s0 > 0 else 0,
})

per_agent_df = pd.DataFrame(per_agent_rows)
display(per_agent_df.to_string(index=False, float_format='{:.1f}'.format))

per_agent_df.to_csv(os.path.join(OUT_DIR, 'attrition_per_agent.csv'), index=False)
print('Saved attrition_per_agent.csv')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. SURVIVORSHIP BIAS CHECK
#    Compare retained (S3) vs excluded functions on: CC, SLOC, doc_lines, doc rate
# ─────────────────────────────────────────────────────────────────────────────
print('\n=== SURVIVORSHIP BIAS CHECK ===')

# Build retained / excluded masks for each cohort
all_turn_mask = pd.Series([True] * len(df_raw), index=df_raw.index)
for col in TURNOVER_COLS:
    all_turn_mask = all_turn_mask & df_raw[f'_{col}_num'].notna()

has_doc_mask = df_raw['doc_entropy'].notna() & (df_raw['doc_lines'] > 0)

def survivorship_profile(cohort_mask, label, retained_mask):
    """Return median CC, SLOC, doc_lines for retained and excluded within cohort."""
    cohort   = df_raw[cohort_mask]
    retained = df_raw[cohort_mask &  retained_mask]
    excluded = df_raw[cohort_mask & ~retained_mask]

    rows = []
    for grp_df, grp_name in [(retained, 'Retained'), (excluded, 'Excluded')]:
        rows.append({
            'cohort': label,
            'subset': grp_name,
            'n': len(grp_df),
            'median_CC':   grp_df['cyclomatic_complexity'].median(),
            'median_SLOC': grp_df['sloc'].median(),
            'median_doc_lines': grp_df['doc_lines'].median(),
            'doc_rate':    (grp_df['doc_lines'] > 0).mean(),
        })
    return rows

surv_rows = []
surv_rows += survivorship_profile(df_raw['group'] == 'agent', 'Agent',     all_turn_mask & has_doc_mask)
surv_rows += survivorship_profile(df_raw['group'] == 'human', 'Developer', all_turn_mask & has_doc_mask)

surv_df = pd.DataFrame(surv_rows)
display(surv_df.to_string(index=False, float_format='{:.2f}'.format))

# ── Flag if retained set is systematically simpler or more complex
print()
for cohort in ['Agent', 'Developer']:
    retained = surv_df[(surv_df['cohort'] == cohort) & (surv_df['subset'] == 'Retained')].iloc[0]
    excluded = surv_df[(surv_df['cohort'] == cohort) & (surv_df['subset'] == 'Excluded')].iloc[0]
    cc_diff  = retained['median_CC'] - excluded['median_CC']
    sloc_diff= retained['median_SLOC'] - excluded['median_SLOC']
    print(f"[{cohort}] CC diff (retained - excluded): {cc_diff:+.2f}  SLOC diff: {sloc_diff:+.2f}")
    if abs(cc_diff) > 2 or abs(sloc_diff) > 10:
        direction = 'SIMPLER' if cc_diff < 0 else 'MORE COMPLEX'
        print(f"  *** Retained functions are {direction} — survivorship bias is present ***")
    else:
        print(f"  Retained and excluded are broadly similar in complexity.")

surv_df.to_csv(os.path.join(OUT_DIR, 'attrition_survivorship.csv'), index=False)
print('\nSaved attrition_survivorship.csv')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. COMMIT VELOCITY CONTROL
#    Use pr_date_merged and pr_date_created as a proxy for repository activity.
#    Compare the distribution of PR duration (merged - created) between cohorts.
#    A long PR duration may mean fewer commits / unit time -> turnover metrics
#    using commit offsets (C5, C10, C20) measure longer calendar time for
#    developers than agents if their repos have slower commit cadence.
# ─────────────────────────────────────────────────────────────────────────────
print('\n=== COMMIT VELOCITY PROXY ===')

df_dates = df_raw.copy()
for col in ['pr_date_merged', 'pr_date_created']:
    df_dates[col] = pd.to_datetime(df_dates[col], errors='coerce', utc=True)

df_dates['pr_duration_days'] = (
    (df_dates['pr_date_merged'] - df_dates['pr_date_created']).dt.total_seconds() / 86400
)

# Remove impossible/missing values
df_dates = df_dates[df_dates['pr_duration_days'] >= 0]

for grp, grp_name in [('agent', 'Agent'), ('human', 'Developer')]:
    dur = df_dates[df_dates['group'] == grp]['pr_duration_days'].dropna()
    print(f"[{grp_name}] PR duration (days): "
          f"median={dur.median():.1f}  mean={dur.mean():.1f}  p25={dur.quantile(0.25):.1f}  p75={dur.quantile(0.75):.1f}  n={len(dur)}")

# Mann-Whitney test on PR duration
ag_dur = df_dates[df_dates['group'] == 'agent']['pr_duration_days'].dropna()
hu_dur = df_dates[df_dates['group'] == 'human']['pr_duration_days'].dropna()

if len(ag_dur) > 0 and len(hu_dur) > 0:
    stat, p = mannwhitneyu(ag_dur, hu_dur, alternative='two-sided')
    r = 1 - (2 * stat) / (len(ag_dur) * len(hu_dur))
    print(f"\nMann-Whitney PR duration: U={stat:.0f}, p={p:.3e}, r={r:.4f}")
    if p < 0.05:
        faster = 'Agent' if ag_dur.median() < hu_dur.median() else 'Developer'
        print(f"*** {faster} PRs are merged significantly faster ***")
        print("    This means commit-offset metrics (C5, C10, C20) cover different")
        print("    calendar windows for agents vs developers. Interpret cautiously.")

# Per-repo commit velocity proxy (number of PRs per repo per year)
df_dates['pr_year'] = df_dates['pr_date_created'].dt.year
prs_per_repo = df_dates.groupby(['repo', 'group']).size().reset_index(name='n_prs')

print("\nMedian PRs per repository (proxy for repo activity):")
for grp, grp_name in [('agent', 'Agent'), ('human', 'Developer')]:
    vals = prs_per_repo[prs_per_repo['group'] == grp]['n_prs']
    print(f"  [{grp_name}]: median={vals.median():.0f}  mean={vals.mean():.1f}  max={vals.max()}")

velocity_df = df_dates.groupby('group')['pr_duration_days'].describe().reset_index()
velocity_df.to_csv(os.path.join(OUT_DIR, 'attrition_commit_velocity.csv'), index=False)
print('\nSaved attrition_commit_velocity.csv')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. VISUALISATION — funnel chart
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5), dpi=300)

stages = funnel_df['stage'].tolist()
n_agent_vals = funnel_df['n_agent'].tolist()
n_dev_vals   = funnel_df['n_developer'].tolist()

x = np.arange(len(stages))
width = 0.35

bars_ag = ax.bar(x - width/2, n_agent_vals, width, label='Agent', color='#A7C7E7', edgecolor='grey')
bars_hu = ax.bar(x + width/2, n_dev_vals,   width, label='Developer', color='#BDE5B8', edgecolor='grey')

for bar in bars_ag:
    ax.annotate(f'{int(bar.get_height())}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)
for bar in bars_hu:
    ax.annotate(f'{int(bar.get_height())}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)

short_labels = [s.split(':')[0] for s in stages]
ax.set_xticks(x)
ax.set_xticklabels(short_labels, fontsize=10)
ax.set_ylabel('Number of functions', fontsize=12)
ax.set_title('Attrition Funnel: Agent vs. Developer Cohorts', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'attrition_funnel_chart.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved attrition_funnel_chart.png')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. PR DURATION DISTRIBUTION PLOT
# ─────────────────────────────────────────────────────────────────────────────
plot_dur = df_dates[['group', 'pr_duration_days']].dropna()
plot_dur = plot_dur[plot_dur['pr_duration_days'] <= plot_dur['pr_duration_days'].quantile(0.99)]
plot_dur['cohort'] = plot_dur['group'].map({'agent': 'Agent', 'human': 'Developer'})

fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
sns.boxplot(
    data=plot_dur, x='cohort', y='pr_duration_days',
    palette=PALETTE, showfliers=False, ax=ax,
    order=['Agent', 'Developer']
)
ax.set_title('PR Duration (days) — Proxy for Commit Velocity', fontsize=13, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('PR open duration (days)', fontsize=12)
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'attrition_pr_duration.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved attrition_pr_duration.png')

## Narrative summary for Threats to Validity section

Use the numbers from `attrition_funnel.csv` to populate a paragraph like this:

> *Of the 13,467 functions in the initial dataset, X,XXX (YY%) are agent-authored and Z,ZZZ (WW%) are developer-authored. After requiring documentation quality columns (doc_entropy, doc_code_overlap, doc_redundancy) to be non-missing, N₁ functions remain. Restricting to functions for which all five commit-offset turnover columns (C5, C10, C20, M1, M3) are also available leaves N₂ = 446 functions — a retention rate of R_a% for agents and R_d% for developers.*

> *The [symmetric/asymmetric] attrition [does not/does] differ substantially between cohorts ([Δ] percentage points). Survivorship analysis shows that retained functions have [similar/higher/lower] median CC ([X vs Y]) and SLOC ([X vs Y]) compared to excluded functions, [suggesting/providing no strong evidence for] a survivorship bias towards [simpler/more complex] functions in the turnover sample. Turnover conclusions should therefore be interpreted with caution and may not generalise to highly complex or rarely-modified functions.*

## Revision note

- **What this adds:** Quantifies the 96.7% attrition at each filtering stage, broken down by cohort, so readers and reviewers can see *where* functions are lost and *whether* the loss is symmetric.
- **Where to cite in paper:** Sections 3.4 (Data Collection), 4.4 (RQ4 results), and Threats to Validity (Section 5 or 7). Add the funnel chart as a figure or table in the Threats section.
- **What to watch for:** If agent attrition is substantially higher than developer attrition (e.g., because agent PRs are in fewer repositories and those repos go inactive), the retained agent sample may be from a very narrow set of repositories — reducing generalisability. Flag this explicitly if `pct_agent_retained` < `pct_developer_retained - 10`.